In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.DataFrame({'outlook':['sunny','sunny','overcast','rain','rain','rain','overcast','sunny','sunny','rain','sunny','overcast','overcast','rain'], 
                   'humidity':['high','high','high','high','normal','normal','normal','high','normal','normal','normal','high','normal','high'], 
                   'wind':['weak','strong','weak','weak','weak','strong','strong','weak','weak','weak','strong','strong','weak','strong'], 
                   'tennis': ['n','n','y','y','y','n','y','n','y','y','y','y','y','n']})


In [3]:
X = df.drop('tennis', axis=1).to_numpy().reshape((-1,df.shape[1]-1))
y = df['tennis'].to_numpy()

In [4]:
class DT_Node():
    def __init__(self):
        self.feature = None
        self.gain = None
        self.children = {}
        self.leaf = False
        self.leaf_class = None
    
    def print_info(self):
        if not self.leaf:
            print("feature", self.feature, "with gain:", np.round(self.gain,3))
            # print("Branches: ", list(self.children.keys()))
        else:
            print("class is:", self.leaf_class)

In [5]:
class DT_ID3():
    def __init__(self):
        self.tree = None
        self.classes = None
        self.n_rows = None
        self.n_cols = None
        self.selected_features = []

    def entropy(self, y_rows): 
        entropy = 0 
        for c in self.classes: 
            p = np.sum(y_rows==c) / len(y_rows) 
            if p != 0: 
                # Task 1
                entropy -= p * np.log2(p)
        return entropy

    def get_best_split_gain(self, X, y): 
        if len(np.unique(y)) == 1: # pure subset 
            return None, None
            
        gains = np.zeros(self.n_cols)       
        for j in range(self.n_cols): # loop over features 
            if j in self.selected_features: # feature already used
                gains[j] = 0 
                continue  
            vals = np.unique(X[:,j]) 
            entr_children = 0 
            # Task 2
            for val in vals:
                child_rows = X[:, j] == val
                y_child = y[child_rows]
                entr_children += (
                    len(y_child) / len(y)
                ) * self.entropy(y_child)

            gains[j] = self.entropy(y) - entr_children
        
        if np.all(gains == 0): 
            return None, None 
        
        best = np.argmax(gains) 
        return best, gains[best]  
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.n_rows, self.n_cols = X.shape
        
        # First level 
        root = DT_Node()
        root.feature, root.gain = self.get_best_split_gain(X, y) 
        # root.print_info()
        self.selected_features += [root.feature]
        self.tree = root

        # Rest of tree
        self.create_next_levels(root, X, y)

    def create_next_levels(self, node, X, y): 
        if node.leaf: 
            return 
            
        feature_vals = np.unique(X[:,node.feature])         
        for val in feature_vals: 
            child = DT_Node() 
            # Task 3.a
            child_rows = X[:, node.feature] == val
            X_child = X[child_rows]
            y_child = y[child_rows]

            child.feature, child.gain = self.get_best_split_gain(
                X_child, y_child
            )

            if child.feature == None: 
                child.leaf = True 
                # Task 3.b # ???
                child_classes, child_counts = np.unique(
                    y_child, return_counts=True
                )
                child.leaf_class = child_classes[np.argmax(child_counts)]

            self.selected_features += [child.feature] 
            # Task 3.c
            node.children[str(val)] = child
            self.create_next_levels(child, X_child, y_child)
    
    def classify_sample(self, sample):
        cur_node = self.tree
        while not cur_node.leaf: 
            # Task 4 
            branch = str(sample[cur_node.feature])
            cur_node = cur_node.children[branch]
        return cur_node.leaf_class

    def print_tree(self, node=None, i=1): 
        if not node:
            node = self.tree
        node.print_info() 
        for branch, child in node.children.items(): 
            print('--'*i + '> if ' + branch + ', ', end='') 
            self.print_tree(child, i+1)

In [6]:
model = DT_ID3()
model.fit(X, y)

# Test the entropy method
print(model.entropy(y))

0.9402859586706311


In [7]:
model.print_tree()

feature 0 with gain: 0.247
--> if overcast, class is: y
--> if rain, feature 2 with gain: 0.971
----> if strong, class is: n
----> if weak, class is: y
--> if sunny, feature 1 with gain: 0.971
----> if high, class is: n
----> if normal, class is: y


In [8]:
model.classify_sample(np.array(['rain', 'high', 'strong']))

'n'